In [1]:
import ssl
import certifi
ssl._create_default_https_context = ssl._create_unverified_context
from llama_index.core import SimpleDirectoryReader
import os
from llama_index.core.node_parser import SentenceSplitter
from pinecone import Pinecone
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext, VectorStoreIndex

C:\Users\user1\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_list = [
    "../library-app/docs",
    "../library-app/.kiro", 
    "../library-app/.windsurf"
]
documents = []

for path in path_list:
    if os.path.exists(path):
        try:
            reader = SimpleDirectoryReader(
                input_dir=path, 
                recursive=True, 
                required_exts=[".md"],
                exclude_hidden=False 
            )
            data = reader.load_data()
            
            tool_name = "docs"
            if ".kiro" in path:
                tool_name = "kiro"
            elif ".windsurf" in path:
                tool_name = "windsurf"
                
            for doc in data:
                doc.metadata["tool"] = tool_name
                
            documents.extend(data)
        except ValueError:
            print(f"❓ התיקייה {path} נמצאה, אך לא נמצאו בה קבצי .md")
    else:
        print(f"❌ שגיאה: הנתיב {path} לא נמצא. ודאי ששם התיקייה מדויק!")

print(f"Total documents loaded: {len(documents)}")


Total documents loaded: 6


In [3]:
tool_counts = {}
for doc in documents:
    tool = doc.metadata.get("tool", "unknown")
    tool_counts[tool] = tool_counts.get(tool, 0) + 1

print("📊 סיכום קבצים לפי כלי (Metadata):")
for tool, count in tool_counts.items():
    print(f" - {tool}: {count} קבצים")



📊 סיכום קבצים לפי כלי (Metadata):
 - docs: 4 קבצים
 - kiro: 1 קבצים
 - windsurf: 1 קבצים


In [4]:
node_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=20)
nodes = node_parser.get_nodes_from_documents(documents)

In [5]:
from dotenv import load_dotenv
import os

load_dotenv() 

print(f"Is API Key loaded? {bool(os.getenv('COHERE_API_KEY'))}")

Is API Key loaded? True


In [6]:
from llama_index.embeddings.cohere import CohereEmbedding
import os
from llama_index.core import Settings
embed_model = CohereEmbedding(
    api_key=os.getenv("COHERE_API_KEY"),
    model_name="embed-multilingual-v3.0",
)
Settings.embed_model = embed_model

In [7]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pinecone_index = pc.Index("agentic-coding-index")
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes, storage_context=storage_context)

2026-03-14 20:29:24,605 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 6/6 [00:04<00:00,  1.37it/s]


# Part 1: Basic RAG with Gradio Interface

In [8]:
from llama_index.llms.cohere import Cohere
Settings.llm = Cohere(
    api_key=os.getenv("COHERE_API_KEY"), 
    model="command-a-03-2025" 
)

In [9]:
import gradio as gr

query_engine = index.as_query_engine()

def predict(message, history):
    response = query_engine.query(message)
    return str(response)

demo = gr.ChatInterface(
    fn=predict, 
    title="Agentic RAG Assistant (Local Mode)",
    description="שאל אותי כל דבר על התיעוד של הפרויקט!",
)

demo.launch(share=False, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860


2026-03-14 20:29:40,306 - INFO - HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-14 20:29:40,355 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
2026-03-14 20:29:40,557 - INFO - HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


# Part 2: Event-Driven RAG Workflow

In [10]:
from llama_index.core.workflow import Event
from llama_index.core.schema import NodeWithScore

class RetrievedEvent(Event):
    nodes: list[NodeWithScore]
    query: str

class ValidationEvent(Event):
    query: str
    nodes: list[NodeWithScore]

class RetryRetrievalEvent(Event):
    query: str



In [11]:
from llama_index.core import Settings, get_response_synthesizer
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step, Context


Settings.llm = Cohere(api_key=os.getenv("COHERE_API_KEY"), model="command-a-03-2025")
response_synthesizer = get_response_synthesizer(response_mode="compact")

2026-03-14 20:29:40,871 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


In [12]:
class RAGWorkflow(Workflow):
    retries = 0 
    @step
    async def ingest_and_validate(self, ctx: Context, ev: StartEvent) -> RetrievedEvent | StopEvent:
        query = ev.get("query")
        if not query or query.strip() == "":
            return StopEvent(result="⚠️ השאילתה ריקה, נא להזין שאלה.")
        
        print(f"🔎 [ניסיון {self.retries + 1}] מבצע Retrieval עבור: {query}")
        nodes = retriever.retrieve(query)
        return RetrievedEvent(nodes=nodes, query=query)

    @step
    async def validate_results(self, ctx: Context, ev: RetrievedEvent) -> ValidationEvent | RetryRetrievalEvent | StopEvent:
        top_score = ev.nodes[0].score if ev.nodes and ev.nodes[0].score else 0
        
        if top_score < 0.5 and self.retries < 2:
            print(f"🔄 Confidence גבולי ({top_score:.2f}). מנתב לשיפור שאילתה...")
            self.retries += 1
            return RetryRetrievalEvent(query=f"Detailed technical information about {ev.query}")
        
        if not ev.nodes or top_score < 0.35:
            self.retries = 0 # איפוס לריצה הבאה
            return StopEvent(result="❌ לא נמצא מידע מהימן מספיק במסמכים.")

        print(f"✅ ולידציה עברה (Score: {top_score:.2f})")
        self.retries = 0 
        return ValidationEvent(nodes=ev.nodes, query=ev.query)

    @step
    async def handle_retry(self, ctx: Context, ev: RetryRetrievalEvent) -> StartEvent:
        return StartEvent(query=ev.query)
    
    @step
    async def synthesize(self, ctx: Context, ev: ValidationEvent) -> StopEvent:
        if "json" in ev.query.lower() or "extract" in ev.query.lower():
            print("✍️ מחלץ מידע למבנה JSON מובנה (שלב ג')...")
            structured_llm = Settings.llm
            response = structured_llm.structured_predict(
                ProjectDataSchema, 
                prompt_template="Extract technical data from: {context_str}",
                context_str="\n".join([n.text for n in ev.nodes])
            )
            return StopEvent(result=response.model_dump_json(indent=2))
        
        else:
            print("✍️ מנסח תשובה טקסטואלית רגילה (שלב ב')...")
            response = response_synthesizer.synthesize(query=ev.query, nodes=ev.nodes)
            return StopEvent(result=str(response))


In [13]:

retriever = index.as_retriever(similarity_top_k=3)

rag_workflow = RAGWorkflow(timeout=60, verbose=True)
result = await rag_workflow.run(query="What is the primary color of the design system ??")
print("-" * 30)
print(f"🤖 תשובה מה-Workflow: {result}")

2026-03-14 20:29:41,388 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


🔎 [ניסיון 1] מבצע Retrieval עבור: What is the primary color of the design system ??


2026-03-14 20:29:41,685 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


🔄 Confidence גבולי (0.47). מנתב לשיפור שאילתה...
🔎 [ניסיון 2] מבצע Retrieval עבור: Detailed technical information about What is the primary color of the design system ??


2026-03-14 20:29:42,921 - INFO - HTTP Request: POST https://api.cohere.com/v2/embed "HTTP/1.1 200 OK"


✅ ולידציה עברה (Score: 0.50)
✍️ מנסח תשובה טקסטואלית רגילה (שלב ב')...


2026-03-14 20:29:45,552 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


------------------------------
🤖 תשובה מה-Workflow: The primary color of the design system is Ocean Blue, represented by the hexadecimal code #0077be.


In [14]:
async def predict(message, history):
    workflow = RAGWorkflow(timeout=120)
    result = await workflow.run(query=message)
    return str(result)

demo = gr.ChatInterface(
    fn=predict,
    title="Kiro Agentic RAG - Event Driven",
    description="מערכת RAG מבוססת אירועים עם שיפור שאילתות אוטומטי (שלב ב')"
)

demo.launch(share=True, debug=True)

2026-03-14 20:29:45,793 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861


2026-03-14 20:29:46,246 - INFO - HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-14 20:29:46,479 - INFO - HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"
2026-03-14 20:29:46,965 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
2026-03-14 20:29:48,138 - INFO - HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026-03-14 20:29:49,008 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
2026-03-14 20:29:49,111 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


Keyboard interruption in main thread... closing server.


# Part 3: Agentic RAG (Final Stage)

In [15]:
from pydantic import BaseModel, Field
from typing import List

class DecisionItem(BaseModel):
    id: str = Field(description="Unique ID like dec-001")
    title: str = Field(description="Brief title of the decision")
    summary: str = Field(description="What was decided?")
    file: str = Field(description="The source file name")
    line_range: str = Field(description="Approximate line numbers or section name")
    last_modified: str = Field(description="Date of the decision (e.g. 2026-03-12)")

class RuleItem(BaseModel):
    id: str = Field(description="Unique ID like rule-001")
    description: str = Field(description="The specific rule, guideline, or configuration")
    category: str = Field(description="Category like UI, Database, API, etc.")
    source_file: str = Field(description="The filename where this rule was found")
    observed_at: str = Field(description="The date this rule was documented")

class WarningItem(BaseModel):
    id: str = Field(description="Unique ID like warn-001")
    message: str = Field(description="The warning message or restriction")
    severity: str = Field(description="Severity level (e.g., High, Low)")
    source_file: str = Field(description="The filename where this warning was found")

class ProjectDataSchema(BaseModel):
    decisions: List[DecisionItem]
    rules: List[RuleItem]
    warnings: List[WarningItem]

In [16]:
import json
from llama_index.core.llms import ChatMessage

async def extract_and_save_offline_data():
    print("⏳ מחלץ נתונים בשיטה בטוחה (ללא Pydantic Errors)...")
    
    context_text = ""
    for doc in documents[:3]:
        f_name = doc.metadata.get('file_name', 'doc.md')
        context_text += f"\n[SOURCE: {f_name}]\n{doc.text}\n"

    prompt = f"""
    You are a JSON extractor. Based on the text below, extract technical items.
    Return ONLY a valid JSON object with this exact keys:
    {{
      "decisions": [{{ "id": "d1", "title": "...", "summary": "...", "file": "...", "last_modified": "2026-03-14" }}],
      "rules": [{{ "id": "r1", "description": "...", "category": "...", "source_file": "...", "observed_at": "2026-03-14" }}],
      "warnings": [{{ "id": "w1", "message": "...", "severity": "High", "source_file": "..." }}]
    }}
    
    TEXT:
    {context_text[:15000]}
    """
    
    response = Settings.llm.chat([ChatMessage(role="user", content=prompt)])
    raw_content = str(response).strip()
    
    try:
        if "```json" in raw_content:
            raw_content = raw_content.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_content:
            raw_content = raw_content.split("```")[1].split("```")[0].strip()
            
        json_data = json.loads(raw_content)
        
        with open("extracted_data.json", "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=2, ensure_ascii=False)
        print("✅ הצלחה! הקובץ extracted_data.json נוצר עם נתונים אמיתיים.")
        
    except Exception as e:
        print(f"❌ שגיאה בפענוח ה-JSON: {e}")
        print("מנסה לשמור את התוכן הגולמי בכל זאת...")
        with open("extracted_data.json", "w", encoding="utf-8") as f:
            f.write(raw_content)

await extract_and_save_offline_data()

⏳ מחלץ נתונים בשיטה בטוחה (ללא Pydantic Errors)...


2026-03-14 20:30:14,912 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


✅ הצלחה! הקובץ extracted_data.json נוצר עם נתונים אמיתיים.


In [17]:
from llama_index.core.llms import ChatMessage
import os

async def agent_logic(message: str):
    decision_prompt = f"""
    You are a smart router. Classify the user message into exactly ONE of these three categories:
    
    1. 'GENERAL': Greetings, personal questions, or general knowledge.
    2. 'STRUCTURED': Questions about rules, decisions, or warnings that require specific details like dates, source files, or lists (e.g., "What is the latest rule?", "List all UI decisions").
    3. 'SEMANTIC': General technical questions about how the app works or logic ("How does the database connect?").
    
    Message: "{message}"
    Answer ONLY with one word: GENERAL, STRUCTURED, or SEMANTIC.
    """
    
    msg = [ChatMessage(role="user", content=decision_prompt)]
    decision_res = str(Settings.llm.chat(msg)).strip().upper()
    
    if "STRUCTURED" in decision_res:
        print(f"📁 שלב ג: ביצוע Structured Querying מול הנתונים המובנים")
        
        if os.path.exists("extracted_data.json"):
            with open("extracted_data.json", "r", encoding="utf-8") as f:
                data = f.read()
            
            query_engine_prompt = f"""
            You are a Structured Data Engine. Your task is to answer the user's request by querying the provided JSON data.
            Pay close attention to 'observed_at' or 'last_modified' fields if the user asks for 'latest' or 'recent' information.
            
            User Request: {message}
            
            JSON Data:
            {data}
            """
            response = Settings.llm.chat([ChatMessage(role="user", content=query_engine_prompt)])
            return str(response)
        else:
            return "❌ שגיאה: קובץ הנתונים לא נמצא. נא להריץ את שלב החילוץ."
            
    elif "SEMANTIC" in decision_res:
        print(f"🔍 ניתוב למסלול SEMANTIC (RAG חיפוש וקטורי)")
        wf = RAGWorkflow(timeout=120)
        return await wf.run(query=message)
        
    else:
        print(f"💬 ניתוב למסלול GENERAL")
        system_persona = """
        You are the 'Smart Agentic RAG Assistant'. You manage documentation from Kiro, Windsurf, and more. 
        Answer the user proudly about your architecture. Answer in the same language the user spoke.
        """
        msg_gen = [
            ChatMessage(role="system", content=system_persona),
            ChatMessage(role="user", content=message)
        ]
        response = Settings.llm.chat(msg_gen)
        return str(response)

In [18]:
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import FunctionTool

async def technical_research_tool(query: str) -> str:
    """
    Search for technical specifications, UI design decisions, and JSON extraction.
    Use this tool ONLY for technical questions about the library app or when JSON is requested.
    """
    workflow = RAGWorkflow(timeout=120)
    result = await workflow.run(query=query)
    return str(result)

research_tool = FunctionTool.from_defaults(
    async_fn=technical_research_tool,
    name="technical_research",
    description="Extracts technical details or JSON from project documents."
)

from llama_index.core.agent import ReActAgent

agent = ReActAgent(
    tools=[research_tool], 
    llm=Settings.llm, 
    verbose=True
)


In [ ]:
import gradio as gr

async def predict(message, history):
    print(f"\n--- בקשה חדשה מהממשק: {message} ---")
    result = await agent_logic(message)
    return str(result)

demo = gr.ChatInterface(
    fn=predict, 
title="Agentic RAG System - Final Stage",
description="מערכת Agentic RAG מתקדמת, המשלבת ניתוב חכם בין חיפוש וקטורי סמנטי מול Pinecone, חילוץ נתונים מובנים (Offline JSON Extraction), ומודל שפה טבעית.")

demo.launch(share=False, debug=True)

2026-03-14 20:30:21,645 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861


2026-03-14 20:30:21,755 - INFO - HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
2026-03-14 20:30:21,765 - INFO - HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


2026-03-14 20:30:22,034 - INFO - HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
2026-03-14 20:30:22,438 - INFO - HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


...

2026-03-14 20:31:41,800 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"



--- בקשה חדשה מהממשק: Summarize the budget and marketing plan for this project ---


2026-03-14 20:31:46,195 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"


📁 שלב ג: ביצוע Structured Querying מול הנתונים המובנים


2026-03-14 20:31:48,420 - INFO - HTTP Request: POST https://api.cohere.com/v1/chat "HTTP/1.1 200 OK"
